In [2]:
# extract_5step_windows.py
import os
import pandas as pd
import numpy as np
import scipy.signal as signal

resampled_dir = '../data/kaggle-drdataboston/resampled_subjects'
output_csv = '../data/kaggle-drdataboston/all_10step_windows_no_overlap.csv'
fs = 50
samples_per_step = int(fs * 0.6)

rows = []

for filename in os.listdir(resampled_dir):
    if not filename.endswith('.csv'):
        continue

    filepath = os.path.join(resampled_dir, filename)
    df = pd.read_csv(filepath)

    if 'motionUserAccelerationX.G.' not in df.columns:
        continue

    acc_mag = np.sqrt(
        df['motionUserAccelerationX.G.']**2 +
        df['motionUserAccelerationY.G.']**2 +
        df['motionUserAccelerationZ.G.']**2
    )

    b, a = signal.butter(2, [0.5 / (fs/2), 3.0 / (fs/2)], btype='band')
    acc_filt = signal.filtfilt(b, a, acc_mag)

    peaks, _ = signal.find_peaks(acc_filt, distance=fs*0.4)

    for i in range(0, len(peaks) - 10, 10):
        start = peaks[i]
        end = peaks[i + 10]
        window = acc_filt[start:end]

        if len(window) < 30:
            continue

        window_resampled = signal.resample(window, 100)
        row = [filename] + list(window_resampled)
        rows.append(row)

    print(f"{filename}: {len(rows)} windows total")

# Save to CSV
columns = ['filename'] + [f'f{i}' for i in range(100)]
pd.DataFrame(rows, columns=columns).to_csv(output_csv, index=False)
print(f"✅ Saved all 10-step windows to {output_csv}")


sub58-lw-s1.csv: 34 windows total
sub92-lw-s2.csv: 65 windows total
sub93-rp-s1.csv: 100 windows total
sub56-lw-s2.csv: 127 windows total
sub43-rp-s2.csv: 157 windows total
sub32-lw-s2.csv: 188 windows total
sub20-rp-s2.csv: 223 windows total
sub38-lw-s1.csv: 267 windows total
sub26-rp-s1.csv: 301 windows total
sub79-lw-s2.csv: 336 windows total
sub21-rp-s2.csv: 371 windows total
sub33-rp-s1.csv: 403 windows total
sub37-lw-s1.csv: 438 windows total
sub20-rp-s1.csv: 470 windows total
sub40-lw-s1.csv: 506 windows total
sub24-lw-s1.csv: 546 windows total
sub62-lw-s2.csv: 573 windows total
sub31-lw-s2.csv: 603 windows total
sub31-rp-s2.csv: 632 windows total
sub67-lw-s2.csv: 661 windows total
sub13-lw-s1.csv: 691 windows total
sub28-rp-s2.csv: 723 windows total
sub83-rp-s1.csv: 751 windows total
sub83-rp-s2.csv: 779 windows total
sub24-lw-s2.csv: 810 windows total
sub18-lw-s2.csv: 845 windows total
sub82-rp-s2.csv: 877 windows total
sub50-rp-s1.csv: 924 windows total
sub18-rp-s1.csv: 965 w

In [ ]:
# train_autoencoder.py
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models
import joblib

windows_df = pd.read_csv('../data/kaggle-drdataboston/all_10step_windows_no_overlap.csv')
matrix_df = pd.read_csv('../data/kaggle-drdataboston/matrix.csv')

# Match metadata to each window
meta_columns = ['Weight', 'Age', 'Height (CM)', 'gender']
file_columns = [
    'subject_left_waist_session1',
    'subject_right_pocket_session1',
    'subject_left_waist_session2',
    'subject_right_pocket_session2'
]

metadata = []
for filename in windows_df['filename']:
    meta_row = None
    for _, row in matrix_df.iterrows():
        if filename in row[file_columns].values:
            meta_row = row
            break
    if meta_row is None:
        metadata.append([np.nan] * len(meta_columns))
    else:
        metadata.append([meta_row[col] for col in meta_columns])

meta_df = pd.DataFrame(metadata, columns=meta_columns)
data_df = pd.concat([windows_df.drop(columns='filename'), meta_df], axis=1)
data_df.dropna(inplace=True)

X = data_df.iloc[:, :100].values
meta_out = data_df[meta_columns]

# Normalize input
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Autoencoder
latent_dim = 16
input_dim = X.shape[1]

input_layer = layers.Input(shape=(input_dim,))
encoded = layers.Dense(64, activation='relu')(input_layer)
encoded = layers.Dense(64, activation='relu')(encoded)
encoded = layers.Dense(32, activation='relu')(encoded)
latent = layers.Dense(latent_dim, activation='relu')(encoded)
decoded = layers.Dense(32, activation='relu')(latent)
decoded = layers.Dense(64, activation='relu')(decoded)
decoded = layers.Dense(64, activation='relu')(decoded)
output_layer = layers.Dense(input_dim, activation='linear')(decoded)

autoencoder = models.Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.fit(X_scaled, X_scaled, epochs=50, batch_size=32, shuffle=True)

# Save encoder model separately
encoder = models.Model(inputs=input_layer, outputs=latent)
encoder.save("step_encoder_10step_no_overlap.keras")
joblib.dump(scaler, "step_scaler_10step_no_overlap.pkl")

print(f"Autoencoder trained and saved as step_encoder_10step_no_overlap.keras")


2025-04-03 17:01:08.682191: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-03 17:01:08.682738: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-03 17:01:08.685162: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-03 17:01:08.690717: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743688868.700356   61475 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743688868.70

Epoch 1/50


2025-04-03 17:02:07.521246: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


280/280 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.8831 
Epoch 2/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.7042  
Epoch 3/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5064
Epoch 4/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.4442  
Epoch 5/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3534  
Epoch 6/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.4208  
Epoch 7/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3038
Epoch 8/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3037
Epoch 9/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 978us/step - loss: 0.3189
Epoch 10/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3010
Epoch 11/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2765
Epoch 12/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.3218
Epoch 13/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2570 
Epoch 14/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2871
Epoch 15/50
280/280 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - l

Encoded shape: (342, 16)


/home/robert/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
